In [1]:
import os, random, gc, torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from scipy.special import softmax
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, 
    TrainingArguments, Trainer, DataCollatorWithPadding
)

# --- CONFIG ---
MODEL_NAME = "distilbert/distilroberta-base"
MAX_LEN = 256 # Más texto = mejor comprensión
SEED = 42

# --- DATA CON MÁS CONTEXTO ---
train_df = pd.read_csv("../../data/train.csv")
test_df = pd.read_csv("../../data/test_nolabel.csv")
train_df = train_df[train_df['label'].astype(str).isin(['0', '1'])].copy()
train_df['label'] = train_df['label'].astype(int)

def build_text_pro(df):
    df = df.copy()
    # Llenamos nulos con "none"
    for c in ["statement", "subject", "speaker", "party_affiliation"]:
        df[c] = df[c].fillna("none").astype(str).str.lower()
    
    # Estructura de texto más rica
    df["model_text"] = (
        "speaker: " + df["speaker"] + " (" + df["party_affiliation"] + ") " +
        "topic: " + df["subject"] + " </s> " + 
        "statement: " + df["statement"]
    )
    return df

train_df = build_text_pro(train_df)
test_df = build_text_pro(test_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def encode(batch): return tokenizer(batch["model_text"], truncation=True, max_length=MAX_LEN)

tr_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df["label"], random_state=SEED)
train_ds = Dataset.from_pandas(tr_df[["model_text", "label"]]).map(encode, batched=True)
val_ds = Dataset.from_pandas(val_df[["model_text", "label"]]).map(encode, batched=True)
test_ds = Dataset.from_pandas(test_df[["model_text"]]).map(encode, batched=True)

# --- ENTRENAMIENTO ROBUSTO ---
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir="outputs_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1.5e-5,        # Un poco más lento para no pasarse de largo
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=2, 
    num_train_epochs=5,           # Una época más para pulir detalles
    weight_decay=0.05,            # Más regularización para evitar overfitting
    warmup_ratio=0.1,             # Inicio suave
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer)
)

print("\n🚀 Entrenando versión PRO... ¡A por ese 0.63+!")
trainer.train()

# --- UMBRAL ÓPTIMO ---
val_logits = trainer.predict(val_ds).predictions
val_probs = softmax(val_logits, axis=1)[:, 1]

best_t, best_f1 = 0.5, 0
for t in np.arange(0.3, 0.7, 0.01):
    f1 = f1_score(val_df["label"], (val_probs >= t).astype(int), average="macro")
    if f1 > best_f1:
        best_f1 = f1; best_t = t

# --- SUBMISSION ---
test_probs = softmax(trainer.predict(test_ds).predictions, axis=1)[:, 1]
final_preds = (test_probs >= best_t).astype(int)
pd.DataFrame({"id": test_df["id"], "label": final_preds}).to_csv("../../data/submission_pro.csv", index=False)

print(f"✅ ¡LISTO! F1 Local: {best_f1:.4f} (Umbral: {best_t:.2f})")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 101/101 [00:00<00:00, 8840.25it/s]
RobertaForSequenceClassification LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING


🚀 Entrenando versión PRO... ¡A por ese 0.63+!


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,0.599817
2,1.273149,0.614739
3,1.166070,0.639045
4,1.050687,0.669058
5,0.898737,0.720917


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.23it/s]
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.51it/s]
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.84it/s]
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shard

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


✅ ¡LISTO! F1 Local: 0.6400 (Umbral: 0.56)
